# Weather Type Prediction Using Machine Learning
**Type:** Data Analysis | Machine Learning | Predictive Analytics  
**Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn  

---
## Executive Summary
This project investigates meteorological patterns across 13,200 observation records to classify weather into four distinct categories: **Sunny**, **Cloudy**, **Rainy**, and **Snowy**.
The end-to-end pipeline includes:
1. **Exploratory Data Analysis (EDA):** Uncovering relationships between atmospheric features and weather types.
2. **Data Preprocessing & Feature Selection:** Outlier treatment, domain-driven feature engineering, and mutual information scoring.
3. **Machine Learning Benchmarking:** 5-fold cross-validation across 5 classifiers, hyperparameter tuning, and decision-support simulation.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plots
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
print("Libraries successfully imported.")

## 1. Dataset Loading & Initial Inspection
We load the dataset downloaded via Kagglehub (`nikhil7280/weather-type-classification`).

In [ ]:
df = pd.read_csv("data/weather_classification_data.csv")
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing values per column:\n", df.isnull().sum())
print("\nTarget Class Distribution:\n", df["Weather Type"].value_counts())

## 2. Exploratory Data Analysis & Visualizations
Examine class distributions and how key numerical features vary across weather types.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
palette = {"Sunny": "#FDB813", "Cloudy": "#95A5A6", "Rainy": "#3498DB", "Snowy": "#9B59B6"}
sns.countplot(data=df, x="Weather Type", palette=palette, ax=axes[0], edgecolor="black")
axes[0].set_title("Balanced Class Distribution")
sns.boxplot(data=df, x="Weather Type", y="Temperature", palette=palette, ax=axes[1])
axes[1].set_title("Temperature by Weather Type")
plt.tight_layout()
plt.show()

## 3. Preprocessing, Feature Engineering & Selection
We apply:
- Physical bounds clipping for sensor outliers
- Meteorological feature engineering: Dew Point Approximation, Temperature-Humidity Index (THI), Pressure Anomaly, and Freezing flag.
- ColumnTransformer scaling & One-Hot Encoding.

In [ ]:
from src.preprocess import prepare_data

data_bundle = prepare_data("data/weather_classification_data.csv")
print("Training set samples:", data_bundle["X_train_proc"].shape[0])
print("Test set samples:    ", data_bundle["X_test_proc"].shape[0])
print("Total Features:      ", len(data_bundle["feature_names"]))
data_bundle["feature_importance_df"].head(10)

## 4. Model Training, 5-Fold Cross Validation & Benchmark
We evaluate Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, and SVM.

In [ ]:
comparison_df = pd.read_csv("reports/model_comparison.csv")
display(comparison_df)

## 5. Final Model Evaluation & Diagnostics
Evaluation of the champion tuned model on the held-out test set.

In [ ]:
with open("reports/classification_report.txt", "r") as f:
    print(f.read())

## 6. Practical Decision-Support System
Translate predictions into actionable operational advice for Aviation, Agriculture, and Logistics.

In [ ]:
from src.predict import WeatherPredictor

predictor = WeatherPredictor("models/weather_classifier_pipeline.joblib")

sample_weather = {
    "Temperature": -5.0,
    "Humidity": 89,
    "Wind Speed": 22.0,
    "Precipitation (%)": 82.0,
    "Cloud Cover": "overcast",
    "Atmospheric Pressure": 994.0,
    "UV Index": 1,
    "Season": "Winter",
    "Visibility (km)": 2.0,
    "Location": "mountain"
}

result = predictor.predict_with_decision_support(sample_weather)
print("Prediction:", result["prediction_summary"]["predicted_weather"])
print("Confidence:", result["prediction_summary"]["confidence_score"])
print("\nDecision Support Guidance:")
for domain, notes in result["decision_support"].items():
    print(f"- {domain}:")
    for n in notes:
        print(f"    * {n}")